# 01. Load, validate and clean

First step for every new export. It reads the CSVs, runs every validation check, and shows exactly what the cleaning step removes. **Dummy data by default.**

Checks are *hard* (the analysis refuses to run: the derived numbers would be untrustworthy) or *soft* (reported and handled explicitly). Try `DATA_DIR = ROOT / 'data' / 'dummy_dirty'` below to see the validator catch 8 injected faults.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)

DATA_DIR = ROOT / "data" / "dummy"      # <- point this at data/raw/<your export folder> for real data
OUT_DIR = ROOT / "outputs"

from pipeline.io import load_dataset
from pipeline.validate import validate
from pipeline.clean import clean

ds = load_dataset(DATA_DIR)
print({"events": len(ds.events), "trials": ds.events.trial_id.nunique(),
       "participants": ds.participants.participant_id.nunique(), "excluded in exclusions.csv": len(ds.exclusions)})
ds.events.head(10)

## Validation report

In [ ]:
report = validate(ds)
r = report.to_frame()
r

In [ ]:
print('HARD failures:', [x.name for x in report.hard_failures])
print('soft failures:', [x.name for x in report.soft_failures])

### Offending rows
Empty when everything passes. Fix hard failures in the raw export (and note the fix in the cleaning log) before continuing.

In [ ]:
report.details_frame()

## Cleaning
Only three things are ever removed, each logged: exact duplicate event rows, practice trials, and trials listed in `exclusions.csv` (device/app faults, decided without looking at the pattern). **No statistical outlier removal.**

In [ ]:
report.raise_if_hard()        # stops here if the data has hard problems
clean_ds, log = clean(ds)
print(f'{ds.events.trial_id.nunique()} trials before, {clean_ds.events.trial_id.nunique()} after')
log

In [ ]:
log.groupby('step').agg(trials=('trial_id', 'nunique'), event_rows=('n_event_rows', 'sum'))